# Occlusion warning v2 on the 40 dev videos (Kaggle, self-contained)

The V-JEPA predictor as an early occlusion warning for the best variant (re-localization verified by OSTrack,
score >= 0.6): while warned the target memory is frozen (`src/tracker/relocalize.py`). v2 fixes the
calibration of v1 (which was on for 37% of the checks): every feature is ranked among the video's last
200 checks and the warning fires on the top 5%, for 16 frames.

- cache - V-JEPA maps + predictor surprise maps (kept in the output, ~150 KB/video) - ~2 h
- w_score - OSTrack's score drop only - control without the predictor - ~30 min
- w_both - + predictor surprise at the target - does the predictor add anything - ~30 min
- w_local - + surprise at the target minus the frame's median - without camera motion / lighting - ~30 min
- w_rise - + local surprise and its increase - the onset - ~30 min
- w_online - + surprise with only the target's box hidden (predictor run during tracking) - ~45 min
- w_both_relax - w_both, and moves are easier while warned - freezing vs. relaxing - ~30 min
- w_online_future - w_online + 2 candidates from the predicted future frame - predictor as a source of candidates - ~45 min
- w_learned - logistic regression fitted on train videos (only if `configs/warning_weights.json` exists) - ~30 min

Output: `/kaggle/working/warning_v2.zip`. Settings: Accelerator = GPU T4 x2, Internet = On, no inputs. Then Save Version -> Save & Run All.

In [ ]:
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On, then run again.')
import torch
if torch.cuda.device_count() == 0:
    raise SystemExit('NO GPU: Settings (right panel) -> Accelerator = GPU T4 x2, then run again.')
print(torch.cuda.device_count(), 'GPUs:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown "transformers>=4.53"
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar

In [ ]:
import os, glob, subprocess, sys, numpy as np
os.chdir('/kaggle/working/Object-Tracking')
sys.path.insert(0, '.')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.makedirs('/kaggle/tmp', exist_ok=True)
W = '/kaggle/working'
CKPT = f'{W}/OSTrack_ep0300.pth.tar'
CACHE = '/kaggle/tmp/cache/vjepa'
SPLIT, EVERY, ZIP = 'configs/splits/dev_occlusion.txt', 8, 'warning_v2.zip'
N_VIDEOS = sum(1 for s in open(SPLIT) if s.strip())
BEST = ['--reloc-verify', '3', '--verify-min', '0.6']    # the best dev variant
N_GPU = max(torch.cuda.device_count(), 1)
os.makedirs(f'{W}/logs', exist_ok=True)

def run_shards(name, args):
    procs = []
    for i in range(N_GPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'{W}/logs/{name}_shard{i}.log', 'w')
        procs.append(subprocess.Popen(['python'] + args + ['--shard', str(i), '--num-shards', str(N_GPU)],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    os.system(f"grep -hE 'FAILED|Error|Traceback' {W}/logs/{name}_shard*.log | head -n 20")
    os.system(f"tail -n 1 {W}/logs/{name}_shard*.log")
    print(name, 'exit codes', codes)

def pack():
    os.system(f'cd {W} && rm -f {ZIP} && zip -qr {ZIP} results_raw logs 2>/dev/null; ls -la {W}/{ZIP}')

def track(name, extra):
    """The best variant + `extra` on SPLIT -> results_raw/<name>/lasot."""
    run_shards(name, ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', SPLIT, '--vjepa-cache', CACHE,
                      '--out', f'{W}/results_raw/{name}/lasot', '--work', '/kaggle/tmp'] + BEST + extra)
    pack()

## Self-test

In [ ]:
# 30 frames, random V-JEPA maps: every warning mode, the online predictor and future candidates run
os.makedirs('/kaggle/tmp/selftest/cache', exist_ok=True)
np.savez('/kaggle/tmp/selftest/cache/airplane-1.npz', frames=np.arange(0, 3000, EVERY),
         maps=np.random.randn(len(range(0, 3000, EVERY)), 16, 16, 1024).astype(np.float16),
         surprise=np.random.rand(len(range(0, 3000, EVERY)), 16, 16).astype(np.float16))
base = ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', 'airplane-1', '--max-frames', '30',
        '--vjepa-cache', '/kaggle/tmp/selftest/cache', '--work', '/kaggle/tmp/selftest/work'] + BEST
for name, extra in [('off', []), ('score', ['--occlusion-warning', 'score']), ('both', ['--occlusion-warning', 'both']),
                    ('rise', ['--occlusion-warning', 'rise', '--warn-relax']),
                    ('online', ['--occlusion-warning', 'online', '--online-predictor', '--future-candidates', '2'])]:
    subprocess.run(['python'] + base + ['--out', f'/kaggle/tmp/selftest/{name}'] + extra, check=True)
import pandas as pd
log = pd.read_csv('/kaggle/tmp/selftest/online/airplane-1_reloc.csv')
assert log.online.notna().all(), 'online predictor surprise missing'
print('SELF-TEST OK')

## V-JEPA cache with the predictor's surprise maps

In [ ]:
for attempt in range(2):  # a re-run only does the videos that failed to download
    run_shards('vjepa_cache', ['scripts/cache_vjepa.py', '--sequences', SPLIT, '--out', CACHE, '--work', '/kaggle/tmp',
                               '--every', str(EVERY), '--surprise', '--surprise-out', f'{W}/results_raw/surprise_maps'])
    n = len(glob.glob(f'{CACHE}/*.npz'))
    print(n, 'of', N_VIDEOS, 'videos cached')
    if n == N_VIDEOS:
        break
assert n == N_VIDEOS, 'V-JEPA cache incomplete'
pack()

## Runs

In [ ]:
RUNS = [('w_score', ['--occlusion-warning', 'score']),
        ('w_both', ['--occlusion-warning', 'both']),
        ('w_local', ['--occlusion-warning', 'local']),
        ('w_rise', ['--occlusion-warning', 'rise']),
        ('w_online', ['--occlusion-warning', 'online', '--online-predictor']),
        ('w_both_relax', ['--occlusion-warning', 'both', '--warn-relax']),
        ('w_online_future', ['--occlusion-warning', 'online', '--online-predictor', '--future-candidates', '2'])]
if os.path.exists('configs/warning_weights.json'):
    RUNS.append(('w_learned', ['--occlusion-warning', 'learned', '--warn-weights', 'configs/warning_weights.json',
                               '--online-predictor']))
for name, extra in RUNS:
    track(f'ostrack_vjepa_reloc_{name}', extra)
os.system(f'rm -rf {W}/Object-Tracking {CKPT}')